# Attention Mechanism Math — Theory & Implementations

This notebook covers the complete mathematics of attention in Transformers:
1. **Scaled dot-product attention** — step-by-step from scratch, √dₖ proof
2. **Softmax** — numerically stable implementation, temperature scaling, Jacobian
3. **Masking** — causal, padding, sliding window, combined masks
4. **Multi-head attention (MHA)** — full implementation with output projection
5. **MQA & GQA** — shared-KV variants with parameter comparison
6. **KV cache** — autoregressive inference simulation, memory calculator
7. **RoPE** — rotary positional encoding in attention context
8. **ALiBi** — linear bias attention
9. **Attention analysis** — entropy, attention sinks, rollout, induction patterns
10. **FlashAttention tiling** — conceptual block-tiled implementation
11. **Parameter counting** — exact counts for real models (LLaMA-3, GPT-4 class)
12. **Complexity benchmarking** — O(n²d) vs O(nd²) crossover

All code uses only `numpy` and `matplotlib` (no external ML frameworks).

See [notes.md](notes.md) for full mathematical derivations.

In [ ]:
import numpy as np
np.random.seed(42)

try:
    import matplotlib.pyplot as plt
    import matplotlib
    matplotlib.rcParams['figure.figsize'] = (10, 6)
    matplotlib.rcParams['font.size'] = 11
    HAS_MPL = True
except ImportError:
    HAS_MPL = False
    print("matplotlib not available — text-based outputs only")

def print_matrix(name, M, fmt=".4f"):
    """Pretty-print a matrix with name label."""
    print(f"\n{name} (shape {M.shape}):")
    for row in M:
        print("  [" + ", ".join(f"{x:{fmt}}" for x in row) + "]")

def print_vector(name, v, fmt=".4f"):
    """Pretty-print a vector with name label."""
    print(f"{name}: [{', '.join(f'{x:{fmt}}' for x in v)}]")

---
## 1. Scaled Dot-Product Attention — Step by Step

$$\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

We'll build this from scratch with a tiny example: **n=4 tokens, d=6, dₖ=dᵥ=3**.

Each step:
1. Project input X → Q, K, V via weight matrices
2. Compute raw scores: S = QKᵀ
3. Scale: S̃ = S / √dₖ
4. (Optional) Apply mask
5. Softmax each row → attention weights A
6. Weighted sum: O = AV

In [ ]:
# === Step-by-step Scaled Dot-Product Attention ===
n, d, d_k, d_v = 4, 6, 3, 3

# Input: 4 tokens, each with 6-dim embedding
X = np.random.randn(n, d)
print_matrix("X (input embeddings)", X)

# Learned projection matrices
W_Q = np.random.randn(d, d_k) * 0.5
W_K = np.random.randn(d, d_k) * 0.5
W_V = np.random.randn(d, d_v) * 0.5

# Step 1: Project to Q, K, V
Q = X @ W_Q  # (n, d_k)
K = X @ W_K  # (n, d_k)
V = X @ W_V  # (n, d_v)

print_matrix("Q (queries)", Q)
print_matrix("K (keys)", K)
print_matrix("V (values)", V)

# Step 2: Raw scores — dot product of every query with every key
S = Q @ K.T  # (n, n)
print_matrix("S = QKᵀ (raw scores)", S)

# Step 3: Scale by √d_k
S_scaled = S / np.sqrt(d_k)
print(f"\n√d_k = {np.sqrt(d_k):.4f}")
print_matrix("S̃ = S / √d_k (scaled scores)", S_scaled)

# Step 4: Softmax (numerically stable)
def softmax(x, axis=-1):
    """Numerically stable softmax: subtract max before exp."""
    x_max = np.max(x, axis=axis, keepdims=True)
    e_x = np.exp(x - x_max)
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

A = softmax(S_scaled)  # (n, n) — each row sums to 1
print_matrix("A = softmax(S̃) — attention weights", A)
print(f"\nRow sums (should all be 1.0): {A.sum(axis=1)}")

# Step 5: Weighted aggregation of values
O = A @ V  # (n, d_v)
print_matrix("O = AV — attention output", O)

# Verify: each output row is a convex combination of value rows
print("\n--- Verification ---")
for i in range(n):
    manual = sum(A[i, j] * V[j] for j in range(n))
    print(f"Token {i}: AV = {O[i].round(4)}, manual = {np.array(manual).round(4)}, match = {np.allclose(O[i], manual)}")

In [ ]:
# === Visualise attention weights as a heatmap ===
if HAS_MPL:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Attention weights
    im0 = axes[0].imshow(A, cmap='Blues', vmin=0, vmax=1)
    axes[0].set_title("Attention Weights A", fontsize=13)
    axes[0].set_xlabel("Key position (j)")
    axes[0].set_ylabel("Query position (i)")
    for i in range(n):
        for j in range(n):
            axes[0].text(j, i, f"{A[i,j]:.3f}", ha='center', va='center', fontsize=10)
    plt.colorbar(im0, ax=axes[0], shrink=0.8)
    
    # Scaled scores (pre-softmax)
    im1 = axes[1].imshow(S_scaled, cmap='RdBu_r', vmin=-2, vmax=2)
    axes[1].set_title("Scaled Scores S̃ (pre-softmax)", fontsize=13)
    axes[1].set_xlabel("Key position (j)")
    axes[1].set_ylabel("Query position (i)")
    for i in range(n):
        for j in range(n):
            axes[1].text(j, i, f"{S_scaled[i,j]:.2f}", ha='center', va='center', fontsize=10)
    plt.colorbar(im1, ax=axes[1], shrink=0.8)
    
    plt.tight_layout()
    plt.show()
else:
    print("Attention weights A:")
    print(np.round(A, 3))

---
## 2. √dₖ Scaling — Numerical Proof

**Claim**: If $q_l, k_l \sim \mathcal{N}(0, 1)$ i.i.d., then $\text{Var}(q \cdot k) = d_k$.

We verify empirically: compute dot products for random q, k ∈ ℝᵈₖ across many trials, measure variance, and confirm it equals dₖ. Then show how scaling by √dₖ restores unit variance.

In [ ]:
# === Numerical verification: Var(q·k) = d_k ===
np.random.seed(42)
n_trials = 100_000
dk_values = [1, 4, 16, 64, 128, 256, 512, 1024]

print(f"{'d_k':>6} | {'Var(q·k)':>12} | {'Expected':>10} | {'Var(q·k/√d_k)':>16} | {'Softmax behaviour':>20}")
print("-" * 80)

for dk in dk_values:
    q = np.random.randn(n_trials, dk)
    k = np.random.randn(n_trials, dk)
    
    # Dot product: sum of d_k terms, each product of two N(0,1)
    dots = np.sum(q * k, axis=1)  # (n_trials,)
    
    var_unscaled = np.var(dots)
    var_scaled = np.var(dots / np.sqrt(dk))
    
    # Softmax behaviour: check max probability in a random softmax
    scores_sample = np.random.randn(10, dk)  # pretend 10 queries with dk-dim "broadcast" 
    # Actually let's compute how peaked softmax gets with std = sqrt(dk) inputs
    std = np.sqrt(var_unscaled)
    behaviour = "Nearly one-hot" if std > 5 else "Moderate" if std > 2 else "Smooth"
    
    print(f"{dk:>6} | {var_unscaled:>12.2f} | {dk:>10} | {var_scaled:>16.4f} | {behaviour:>20}")

print(f"\n✓ Var(q·k) ≈ d_k for all values (theory confirmed)")
print(f"✓ Var(q·k/√d_k) ≈ 1.0 for all values (scaling works)")

In [ ]:
# === Visualise softmax saturation with and without scaling ===
np.random.seed(42)

if HAS_MPL:
    fig, axes = plt.subplots(2, 3, figsize=(15, 9))
    dk_demo = [4, 64, 512]
    n_tokens = 8
    
    for col, dk in enumerate(dk_demo):
        Q_demo = np.random.randn(n_tokens, dk)
        K_demo = np.random.randn(n_tokens, dk)
        scores = Q_demo @ K_demo.T  # (8, 8)
        
        # WITHOUT scaling
        A_unscaled = softmax(scores)
        axes[0, col].imshow(A_unscaled, cmap='Blues', vmin=0, vmax=1)
        axes[0, col].set_title(f"Unscaled, d_k={dk}\nmax weight={A_unscaled.max():.4f}")
        axes[0, col].set_xlabel("Key pos")
        axes[0, col].set_ylabel("Query pos")
        
        # WITH scaling
        A_scaled = softmax(scores / np.sqrt(dk))
        axes[1, col].imshow(A_scaled, cmap='Blues', vmin=0, vmax=1)
        axes[1, col].set_title(f"Scaled by √{dk}={np.sqrt(dk):.1f}\nmax weight={A_scaled.max():.4f}")
        axes[1, col].set_xlabel("Key pos")
        axes[1, col].set_ylabel("Query pos")
    
    axes[0, 0].set_ylabel("Unscaled — Query pos")
    axes[1, 0].set_ylabel("Scaled — Query pos")
    plt.suptitle("Softmax saturation: unscaled (top) vs scaled (bottom)", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
    
    print("Top row: without √d_k scaling → attention becomes one-hot as d_k grows")
    print("Bottom row: with √d_k scaling → attention stays well-distributed")
else:
    for dk in [4, 64, 512]:
        Q_d = np.random.randn(8, dk); K_d = np.random.randn(8, dk)
        s = Q_d @ K_d.T
        print(f"d_k={dk}: max(unscaled softmax)={softmax(s).max():.4f}, "
              f"max(scaled softmax)={softmax(s/np.sqrt(dk)).max():.4f}")

---
## 3. Softmax Deep Dive

### Numerical stability, temperature scaling, and the Jacobian

The softmax function maps ℝⁿ → Δⁿ (the probability simplex):
$$\text{softmax}(z)_i = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

Key properties:
- **Shift invariance**: softmax(z) = softmax(z − c) for any constant c → use c = max(z) for stability  
- **Temperature**: softmax(z/τ) → argmax as τ→0, uniform as τ→∞
- **Jacobian**: ∂softmax(z)ᵢ/∂zⱼ = softmax(z)ᵢ(δᵢⱼ − softmax(z)ⱼ)

In [ ]:
# === Softmax: numerical stability demonstration ===

def softmax_naive(z):
    """Naive softmax — overflows for large z."""
    return np.exp(z) / np.sum(np.exp(z))

def softmax_stable(z, axis=-1):
    """Numerically stable softmax — subtract max before exp."""
    z_max = np.max(z, axis=axis, keepdims=True)
    e_z = np.exp(z - z_max)
    return e_z / np.sum(e_z, axis=axis, keepdims=True)

# Test with normal values
z_normal = np.array([1.0, 2.0, 3.0, 4.0])
print("Normal inputs z =", z_normal)
print("  Naive:  ", softmax_naive(z_normal).round(4))
print("  Stable: ", softmax_stable(z_normal).round(4))

# Test with large values (simulates unscaled attention with large d_k)
z_large = np.array([1000.0, 1001.0, 1002.0, 1000.0])
print(f"\nLarge inputs z = {z_large}")
try:
    result = softmax_naive(z_large)
    print(f"  Naive:  {result}")
except Exception as e:
    print(f"  Naive:  OVERFLOW — {e}")
    # Even without exception, might get nan
    with np.errstate(over='ignore', invalid='ignore'):
        result = softmax_naive(z_large)
        print(f"  Naive result:  {result}  (nan/inf = overflow)")
print(f"  Stable: {softmax_stable(z_large).round(4)}")

# Temperature scaling demonstration
z = np.array([2.0, 1.0, 0.5, -1.0])
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0, 100.0]

print(f"\n{'Temp':>6} | {'softmax(z/τ)':>45} | {'Entropy':>8} | {'Max weight':>10}")
print("-" * 80)
for tau in temperatures:
    p = softmax_stable(z / tau)
    entropy = -np.sum(p * np.log(p + 1e-12))
    print(f"{tau:>6.1f} | [{', '.join(f'{x:.4f}' for x in p)}] | {entropy:>8.4f} | {p.max():>10.4f}")
print("\nτ→0: approaches argmax (one-hot on largest score)")
print("τ→∞: approaches uniform distribution (1/n for all)")

In [ ]:
# === Softmax Jacobian ===
# For backpropagation through attention, we need the Jacobian of softmax.
# ∂softmax(z)_i / ∂z_j = softmax(z)_i * (δ_{ij} - softmax(z)_j)

def softmax_jacobian(z):
    """Compute the full Jacobian matrix of softmax."""
    p = softmax_stable(z)
    n = len(p)
    J = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            if i == j:
                J[i, j] = p[i] * (1 - p[j])
            else:
                J[i, j] = -p[i] * p[j]
    return J

# Vectorised: J = diag(p) - p pᵀ
def softmax_jacobian_vec(z):
    p = softmax_stable(z)
    return np.diag(p) - np.outer(p, p)

z_test = np.array([2.0, 1.0, 0.5, -1.0])
J_loop = softmax_jacobian(z_test)
J_vec = softmax_jacobian_vec(z_test)

print("Softmax Jacobian (loop):")
print_matrix("J", J_loop)

print(f"\nLoop == Vectorised: {np.allclose(J_loop, J_vec)}")

# Verify numerically with finite differences
eps = 1e-5
J_numerical = np.zeros_like(J_loop)
p0 = softmax_stable(z_test)
for j in range(len(z_test)):
    z_plus = z_test.copy(); z_plus[j] += eps
    z_minus = z_test.copy(); z_minus[j] -= eps
    J_numerical[:, j] = (softmax_stable(z_plus) - softmax_stable(z_minus)) / (2 * eps)

print(f"Analytical ≈ Numerical: {np.allclose(J_loop, J_numerical, atol=1e-6)}")

# Key property: each row sums to 0 (probability conservation)
print(f"\nRow sums of Jacobian (should be ~0): {J_loop.sum(axis=1).round(10)}")

---
## 4. Masking — Causal, Padding, Sliding Window

Masks are added to scaled scores **before** softmax. Positions with $M_{ij} = -\infty$ get zero attention weight after softmax.

$$\hat{S} = \frac{QK^\top}{\sqrt{d_k}} + M$$

In [ ]:
# === Masking implementations ===
np.random.seed(42)

def causal_mask(n):
    """Upper-triangular mask: position i cannot attend to j > i."""
    mask = np.zeros((n, n))
    mask[np.triu_indices(n, k=1)] = -np.inf  # above diagonal = -inf
    return mask

def padding_mask(seq_len, pad_positions):
    """Column mask: pad positions cannot be attended to."""
    mask = np.zeros((seq_len, seq_len))
    for p in pad_positions:
        mask[:, p] = -np.inf
    return mask

def sliding_window_mask(n, window_size):
    """Token i only attends to [i-w, i+w]."""
    mask = np.full((n, n), -np.inf)
    for i in range(n):
        lo = max(0, i - window_size)
        hi = min(n, i + window_size + 1)
        mask[i, lo:hi] = 0
    return mask

n_demo = 8

# Create example score matrix
Q_demo = np.random.randn(n_demo, 4)
K_demo = np.random.randn(n_demo, 4)
S_demo = Q_demo @ K_demo.T / np.sqrt(4)

print("=== Causal Mask (n=8) ===")
M_causal = causal_mask(n_demo)
A_causal = softmax(S_demo + M_causal)
print("Mask (0=attend, -inf=block):")
print(np.where(M_causal == -np.inf, '  -∞', '   0').astype(str))
print(f"\nA[0,5] (future token) = {A_causal[0,5]:.6f}  (should be 0)")
print(f"A[5,0] (past token)   = {A_causal[5,0]:.6f}  (should be > 0)")

print("\n=== Padding Mask (positions 6,7 are pads) ===")
M_pad = padding_mask(n_demo, [6, 7])
A_pad = softmax(S_demo + M_pad)
print(f"A[3,7] (attend to pad) = {A_pad[3,7]:.6f}  (should be 0)")
print(f"A[3,2] (attend to real) = {A_pad[3,2]:.6f}  (should be > 0)")

print("\n=== Sliding Window Mask (window=2) ===")
M_window = sliding_window_mask(n_demo, 2)
A_window = softmax(S_demo + M_window)
print(f"A[4,0] (outside window) = {A_window[4,0]:.6f}  (should be 0)")
print(f"A[4,3] (inside window)  = {A_window[4,3]:.6f}  (should be > 0)")

print("\n=== Combined: Causal + Sliding Window (w=3) ===")
M_combined = causal_mask(n_demo) + sliding_window_mask(n_demo, 3)
# -inf + anything = -inf (masks compose correctly)
M_combined = np.where(M_combined < -1e10, -np.inf, 0)
A_combined = softmax(S_demo + M_combined)
print(f"Token 5 attends to: positions", 
      [j for j in range(n_demo) if A_combined[5,j] > 1e-6])

In [ ]:
# === Visualise all mask types ===
if HAS_MPL:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10))
    
    masks = [
        ("No Mask", np.zeros((n_demo, n_demo))),
        ("Causal", M_causal),
        ("Padding (pos 6,7)", M_pad),
        ("Sliding Window (w=2)", M_window),
        ("Causal + Window (w=3)", M_combined),
    ]
    
    attns = [softmax(S_demo + m) for _, m in masks]
    
    for idx, ((name, mask), A_m) in enumerate(zip(masks, attns)):
        row, col = idx // 3, idx % 3
        im = axes[row, col].imshow(A_m, cmap='Blues', vmin=0, vmax=0.5)
        axes[row, col].set_title(name, fontsize=12)
        axes[row, col].set_xlabel("Key position")
        axes[row, col].set_ylabel("Query position")
    
    # Show the mask itself in last panel
    mask_visual = np.where(M_combined < -1e10, 1, 0)
    axes[1, 2].imshow(mask_visual, cmap='Reds', vmin=0, vmax=1)
    axes[1, 2].set_title("Combined Mask\n(red = blocked)", fontsize=12)
    axes[1, 2].set_xlabel("Key position")
    axes[1, 2].set_ylabel("Query position")
    
    plt.suptitle("Attention weights under different masking strategies", fontsize=14)
    plt.tight_layout()
    plt.show()
else:
    print("Mask types implemented — see text output above")

---
## 5. Multi-Head Attention (MHA) — Full Implementation

$$\text{MHA}(X) = \text{Concat}(\text{head}_1, \ldots, \text{head}_H) \cdot W^O$$

where $\text{head}_h = \text{Attention}(XW_h^Q, XW_h^K, XW_h^V)$

Key design: $d_k = d_v = d / H$, so total MHA parameters = **4d²** regardless of H.

In [ ]:
# === Multi-Head Attention from scratch ===

class MultiHeadAttention:
    """Full MHA implementation (standard, no KV sharing)."""
    
    def __init__(self, d_model, n_heads):
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.d_v = d_model // n_heads
        
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        # Xavier initialisation for all projection matrices
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        self.W_Q = np.random.randn(n_heads, d_model, self.d_k) * scale
        self.W_K = np.random.randn(n_heads, d_model, self.d_k) * scale
        self.W_V = np.random.randn(n_heads, d_model, self.d_v) * scale
        self.W_O = np.random.randn(n_heads * self.d_v, d_model) * scale
    
    def forward(self, X, mask=None, return_weights=False):
        """
        X: (n, d_model)
        mask: (n, n) or None
        Returns: (n, d_model)
        """
        n = X.shape[0]
        head_outputs = []
        all_weights = []
        
        for h in range(self.n_heads):
            # Per-head projections
            Q_h = X @ self.W_Q[h]  # (n, d_k)
            K_h = X @ self.W_K[h]  # (n, d_k)
            V_h = X @ self.W_V[h]  # (n, d_v)
            
            # Scaled dot-product attention
            scores = Q_h @ K_h.T / np.sqrt(self.d_k)  # (n, n)
            
            if mask is not None:
                scores = scores + mask
            
            weights = softmax(scores)  # (n, n)
            head_out = weights @ V_h   # (n, d_v)
            
            head_outputs.append(head_out)
            all_weights.append(weights)
        
        # Concatenate and project
        concat = np.concatenate(head_outputs, axis=1)  # (n, H*d_v) = (n, d_model)
        output = concat @ self.W_O  # (n, d_model)
        
        if return_weights:
            return output, np.array(all_weights)  # (H, n, n)
        return output
    
    def param_count(self):
        """Count total parameters."""
        q_params = self.n_heads * self.d_model * self.d_k
        k_params = self.n_heads * self.d_model * self.d_k
        v_params = self.n_heads * self.d_model * self.d_v
        o_params = (self.n_heads * self.d_v) * self.d_model
        return q_params + k_params + v_params + o_params

# === Demo ===
np.random.seed(42)
d_model, n_heads, seq_len = 64, 8, 16
mha = MultiHeadAttention(d_model, n_heads)

X = np.random.randn(seq_len, d_model)
mask = causal_mask(seq_len)

output, weights = mha.forward(X, mask=mask, return_weights=True)

print(f"Configuration: d_model={d_model}, n_heads={n_heads}, d_k={mha.d_k}")
print(f"Input shape:  {X.shape}")
print(f"Output shape: {output.shape}")
print(f"Weights shape (H, n, n): {weights.shape}")
print(f"\nTotal MHA parameters: {mha.param_count():,}")
print(f"Expected (4d²):       {4 * d_model**2:,}")
print(f"Match: {mha.param_count() == 4 * d_model**2}")

# Verify causal masking works
print(f"\nCausal masking check:")
print(f"  weights[0, 0, 5] (head 0, query 0 → key 5, future) = {weights[0, 0, 5]:.6f}")
print(f"  weights[0, 5, 0] (head 0, query 5 → key 0, past)   = {weights[0, 5, 0]:.6f}")
print(f"  All future weights zero: {np.all(weights[:, np.triu_indices(seq_len, k=1)[0], np.triu_indices(seq_len, k=1)[1]] == 0)}")

In [ ]:
# === Visualise per-head attention patterns ===
if HAS_MPL:
    fig, axes = plt.subplots(2, 4, figsize=(18, 9))
    for h in range(n_heads):
        row, col = h // 4, h % 4
        im = axes[row, col].imshow(weights[h], cmap='Blues', vmin=0, vmax=0.3)
        axes[row, col].set_title(f"Head {h}", fontsize=12)
        if col == 0:
            axes[row, col].set_ylabel("Query pos")
        if row == 1:
            axes[row, col].set_xlabel("Key pos")
    
    plt.suptitle(f"Attention weights per head (d_model={d_model}, H={n_heads}, causal mask)", fontsize=14)
    plt.tight_layout()
    plt.show()
    
    print("Each head learns different attention patterns.")
    print("Some heads focus locally (nearby tokens), others attend to distant positions.")
else:
    for h in range(n_heads):
        entropy = -np.sum(weights[h] * np.log(weights[h] + 1e-12)) / seq_len
        print(f"Head {h}: avg entropy = {entropy:.3f}")

---
## 6. Multi-Query Attention (MQA) & Grouped Query Attention (GQA)

**MQA**: All heads share **one** K and **one** V projection. Only Q differs per head.  
**GQA**: Heads divided into G groups; each group shares K, V.

$$\text{KV cache reduction} = \frac{H}{G} \quad (\text{MQA} \Leftrightarrow G=1, \; \text{MHA} \Leftrightarrow G=H)$$

In [ ]:
# === MQA and GQA implementations ===

class GroupedQueryAttention:
    """Unified GQA implementation: G=1→MQA, G=H→MHA."""
    
    def __init__(self, d_model, n_heads, n_kv_groups):
        self.d_model = d_model
        self.n_heads = n_heads
        self.n_kv_groups = n_kv_groups  # G
        self.d_k = d_model // n_heads
        self.d_v = d_model // n_heads
        self.heads_per_group = n_heads // n_kv_groups
        
        assert n_heads % n_kv_groups == 0
        
        scale = np.sqrt(2.0 / (d_model + self.d_k))
        # Q: one projection per head (H total)
        self.W_Q = np.random.randn(n_heads, d_model, self.d_k) * scale
        # K, V: one projection per KV group (G total, G ≤ H)
        self.W_K = np.random.randn(n_kv_groups, d_model, self.d_k) * scale
        self.W_V = np.random.randn(n_kv_groups, d_model, self.d_v) * scale
        self.W_O = np.random.randn(n_heads * self.d_v, d_model) * scale
    
    def forward(self, X, mask=None):
        n = X.shape[0]
        head_outputs = []
        
        for h in range(self.n_heads):
            g = h // self.heads_per_group  # which KV group this head belongs to
            
            Q_h = X @ self.W_Q[h]       # per-head Q
            K_g = X @ self.W_K[g]       # shared K for group g
            V_g = X @ self.W_V[g]       # shared V for group g
            
            scores = Q_h @ K_g.T / np.sqrt(self.d_k)
            if mask is not None:
                scores = scores + mask
            
            weights = softmax(scores)
            head_outputs.append(weights @ V_g)
        
        concat = np.concatenate(head_outputs, axis=1)
        return concat @ self.W_O
    
    def param_count(self):
        q = self.n_heads * self.d_model * self.d_k
        k = self.n_kv_groups * self.d_model * self.d_k
        v = self.n_kv_groups * self.d_model * self.d_v
        o = (self.n_heads * self.d_v) * self.d_model
        return q + k + v + o
    
    def kv_cache_size(self, seq_len, bytes_per_elem=2):
        """KV cache memory in bytes."""
        return 2 * seq_len * self.n_kv_groups * self.d_k * bytes_per_elem

# === Compare MHA, GQA variants, MQA ===
np.random.seed(42)
d, H = 64, 8
configs = [
    ("MHA (G=H=8)",   H),  # G = H → standard MHA
    ("GQA (G=4)",     4),
    ("GQA (G=2)",     2),
    ("MQA (G=1)",     1),
]

X = np.random.randn(16, d)
mask = causal_mask(16)

print(f"{'Config':<16} | {'Params':>10} | {'KV Cache (n=16)':>16} | {'KV Cache (n=8192)':>18} | {'KV Reduction':>12}")
print("-" * 85)

for name, G in configs:
    model = GroupedQueryAttention(d, H, G)
    params = model.param_count()
    kv_16 = model.kv_cache_size(16)
    kv_8192 = model.kv_cache_size(8192)
    reduction = f"{H/G:.0f}×" if G < H else "1× (baseline)"
    print(f"{name:<16} | {params:>10,} | {kv_16:>14,} B | {kv_8192:>16,} B | {reduction:>12}")

    # Verify output shape
    out = model.forward(X, mask=mask)
    assert out.shape == (16, d), f"Expected (16, {d}), got {out.shape}"

print("\n✓ All variants produce output with shape (n, d_model)")
print(f"✓ MQA uses {H}× less KV cache than MHA")

---
## 7. KV Cache — Autoregressive Inference Simulation

During generation, we cache K and V from all previous tokens. At each step:
1. Compute Q, K, V for the **new token only**
2. Append K, V to cache
3. Attend new Q to all cached K → softmax → weighted sum of cached V

$$\text{KV Cache (bytes)} = 2 \times n \times d_{\text{kv}} \times L \times \text{bytes\_per\_elem}$$

In [ ]:
# === KV Cache simulation ===
np.random.seed(42)

def autoregressive_generate_with_cache(X_prompt, W_Q, W_K, W_V, W_O, n_generate):
    """
    Simulate autoregressive generation with KV caching.
    Single-head attention for clarity.
    """
    d_model = W_Q.shape[0]
    d_k = W_Q.shape[1]
    
    # Prefill: compute K, V for all prompt tokens at once
    K_cache = X_prompt @ W_K  # (prompt_len, d_k)
    V_cache = X_prompt @ W_V  # (prompt_len, d_v)
    
    # Current sequence representation
    current_seq = X_prompt.copy()
    generated_outputs = []
    cache_sizes = []
    
    for step in range(n_generate):
        # Only the last token needs new Q, K, V
        x_new = current_seq[-1:]  # (1, d_model) — latest token
        
        q_new = x_new @ W_Q  # (1, d_k) — query for new token
        k_new = x_new @ W_K  # (1, d_k)
        v_new = x_new @ W_V  # (1, d_v)
        
        # Append to cache
        K_cache = np.vstack([K_cache, k_new])  # (prompt+step+1, d_k)
        V_cache = np.vstack([V_cache, v_new])  # (prompt+step+1, d_v)
        
        # Attention: new query against ALL cached keys
        scores = q_new @ K_cache.T / np.sqrt(d_k)  # (1, cache_len)
        weights = softmax(scores)                    # (1, cache_len)
        output = weights @ V_cache                   # (1, d_v)
        
        generated_outputs.append(output[0])
        cache_sizes.append(K_cache.nbytes + V_cache.nbytes)
        
        # "Embed" the output as next token (simplified — real models do LM head + sampling)
        new_embedding = output @ W_O[:d_k, :]  # rough projection back
        current_seq = np.vstack([current_seq, new_embedding])
    
    return generated_outputs, cache_sizes

# Setup
d_model, d_k = 32, 16
prompt_len = 4
n_gen = 12

W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1
W_O = np.random.randn(d_k, d_model) * 0.1

X_prompt = np.random.randn(prompt_len, d_model)

outputs, cache_sizes = autoregressive_generate_with_cache(X_prompt, W_Q, W_K, W_V, W_O, n_gen)

print("=== KV Cache Growth During Generation ===")
print(f"{'Step':>4} | {'Cache Length':>12} | {'Cache Size (bytes)':>18} | {'vs Step 1':>10}")
print("-" * 55)
for step, size in enumerate(cache_sizes):
    cache_len = prompt_len + step + 1
    ratio = size / cache_sizes[0]
    print(f"{step:>4} | {cache_len:>12} | {size:>18,} | {ratio:>9.1f}×")

print(f"\nCache grows linearly with sequence length.")
print(f"Final cache: {cache_sizes[-1]:,} bytes for {prompt_len + n_gen} tokens")

In [ ]:
# === KV Cache Memory Calculator for Real Models ===

def kv_cache_gb(d_model, n_layers, n_kv_heads, d_head, seq_len, precision="fp16"):
    """
    Compute KV cache size in GB.
    
    KV cache = 2 (K+V) × seq_len × n_kv_heads × d_head × n_layers × bytes_per_elem
    """
    bytes_per = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1, "int4": 0.5}[precision]
    total_bytes = 2 * seq_len * n_kv_heads * d_head * n_layers * bytes_per
    return total_bytes / (1024**3)

def model_weight_gb(total_params, precision="fp16"):
    bytes_per = {"fp32": 4, "fp16": 2, "bf16": 2, "int8": 1, "int4": 0.5}[precision]
    return total_params * bytes_per / (1024**3)

# Real model configurations
models = [
    # (name, d_model, n_layers, n_kv_heads, d_head, total_params_B)
    ("LLaMA-3 8B",       4096,  32, 8,   128,  8.0),
    ("LLaMA-3 70B",      8192,  80, 8,   128,  70.6),
    ("LLaMA-3.1 405B",   16384, 126, 8,  128,  405.0),
    ("Mistral 7B",       4096,  32, 8,   128,  7.3),
    ("GPT-4 class (est)",12288, 96, 96,  128,  200),
]

seq_lengths = [2048, 8192, 32768, 128000]

print("=== KV Cache Memory for Real Models ===\n")
print(f"{'Model':<20} | {'Weights':>10} |", end="")
for n in seq_lengths:
    print(f" {'n='+str(n//1000)+'K':>10} |", end="")
print()
print("-" * 80)

for name, d, L, n_kv, d_h, params_B in models:
    weight_gb = model_weight_gb(params_B * 1e9)
    print(f"{name:<20} | {weight_gb:>8.1f}GB |", end="")
    for n in seq_lengths:
        kv_gb = kv_cache_gb(d, L, n_kv, d_h, n)
        marker = " ⚠️" if kv_gb > weight_gb else ""
        print(f" {kv_gb:>8.1f}GB{marker} |" if kv_gb < 100 else f" {kv_gb:>7.0f}GB{marker} |", end="")
    print()

print("\n⚠️ = KV cache exceeds model weight size")
print("\nKey insight: at long contexts, KV cache becomes the dominant memory cost.")
print("This is why GQA (fewer KV heads), MLA, and cache compression are critical.")

---
## 8. Rotary Positional Encoding (RoPE) in Attention

RoPE applies a **position-dependent rotation** to Q and K so the dot product naturally depends on relative position:

$$\text{score}(i, j) = (R_i q)^\top (R_j k) = q^\top R_{j-i} k$$

Each 2D block of the rotation matrix $R_m$ rotates by angle $m\theta_i$:

$$R_m^{(i)} = \begin{pmatrix} \cos(m\theta_i) & -\sin(m\theta_i) \\ \sin(m\theta_i) & \cos(m\theta_i) \end{pmatrix}, \quad \theta_i = 10000^{-2i/d}$$

In [ ]:
# === RoPE Implementation ===

def compute_rope_freqs(d, base=10000.0):
    """Compute θ_i = base^(-2i/d) for i = 0, ..., d/2 - 1."""
    freqs = base ** (-np.arange(0, d, 2, dtype=np.float64) / d)
    return freqs  # shape: (d//2,)

def build_rotation_matrix(pos, freqs):
    """Build the block-diagonal rotation matrix R_pos for given position."""
    d = len(freqs) * 2
    R = np.eye(d)
    for i, theta in enumerate(freqs):
        angle = pos * theta
        c, s = np.cos(angle), np.sin(angle)
        idx = 2 * i
        R[idx, idx] = c
        R[idx, idx+1] = -s
        R[idx+1, idx] = s
        R[idx+1, idx+1] = c
    return R

def apply_rope(x, pos, freqs):
    """Apply RoPE rotation to vector x at position pos."""
    d = len(x)
    x_rot = x.copy()
    for i, theta in enumerate(freqs):
        angle = pos * theta
        c, s = np.cos(angle), np.sin(angle)
        idx = 2 * i
        x0, x1 = x[idx], x[idx + 1]
        x_rot[idx] = x0 * c - x1 * s
        x_rot[idx + 1] = x0 * s + x1 * c
    return x_rot

def apply_rope_batch(X, positions, freqs):
    """Apply RoPE to all rows of X at given positions."""
    return np.array([apply_rope(X[i], positions[i], freqs) for i in range(len(X))])

# === Demo ===
d_head = 8
freqs = compute_rope_freqs(d_head)
print(f"d_head = {d_head}")
print(f"θ values: {freqs.round(6)}")

# Build rotation matrices for positions 0-3
q = np.random.randn(d_head)
k = np.random.randn(d_head)
print(f"\nOriginal q = {q.round(4)}")
print(f"Original k = {k.round(4)}")

# Demonstrate relative position property: 
# dot(R_m q, R_n k) should depend only on (m-n)
print("\n=== Relative Position Property ===")
print(f"{'pos_q':>6} {'pos_k':>6} {'offset':>7} {'dot(R_m q, R_n k)':>20}")
print("-" * 45)
for m in range(5):
    for n in range(5):
        q_rot = apply_rope(q, m, freqs)
        k_rot = apply_rope(k, n, freqs)
        dot_val = np.dot(q_rot, k_rot)
        print(f"{m:>6} {n:>6} {m-n:>7} {dot_val:>20.6f}")

# Verify: same offset → same dot product
print("\n=== Verification: same offset (m-n) → same dot product ===")
for offset in [-2, -1, 0, 1, 2]:
    dots = []
    for m in range(max(0, offset), min(5, 5 + offset)):
        n = m - offset
        if 0 <= n < 5:
            q_rot = apply_rope(q, m, freqs)
            k_rot = apply_rope(k, n, freqs)
            dots.append(np.dot(q_rot, k_rot))
    if dots:
        all_same = np.allclose(dots, dots[0])
        print(f"offset={offset:>3}: dots = {[round(d,6) for d in dots[:4]]}, all equal = {all_same}")

In [ ]:
# === RoPE integrated into attention ===
np.random.seed(42)

def attention_with_rope(X, W_Q, W_K, W_V, freqs, mask=None):
    """Scaled dot-product attention with RoPE on Q and K."""
    n = X.shape[0]
    Q = X @ W_Q
    K = X @ W_K
    V = X @ W_V
    
    # Apply RoPE to Q and K
    positions = np.arange(n)
    Q_rot = apply_rope_batch(Q, positions, freqs)
    K_rot = apply_rope_batch(K, positions, freqs)
    
    d_k = Q.shape[1]
    scores = Q_rot @ K_rot.T / np.sqrt(d_k)
    
    if mask is not None:
        scores = scores + mask
    
    weights = softmax(scores)
    output = weights @ V
    
    return output, weights

# Compare: attention WITHOUT vs WITH RoPE
d_model, d_k = 16, 8
n = 6
X = np.random.randn(n, d_model)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1
freqs = compute_rope_freqs(d_k)

# Without RoPE
Q = X @ W_Q; K = X @ W_K; V = X @ W_V
scores_no_rope = Q @ K.T / np.sqrt(d_k)
A_no_rope = softmax(scores_no_rope)

# With RoPE
_, A_with_rope = attention_with_rope(X, W_Q, W_K, W_V, freqs)

# Permutation test: without RoPE, permuting input → permuted output
# With RoPE, this is NOT the case (position matters)
perm = [3, 1, 4, 0, 2, 5]
X_perm = X[perm]
_, A_perm_no_rope = attention_with_rope(X_perm, W_Q, W_K, W_V, np.zeros_like(freqs))  # zero freqs = no rotation
_, A_perm_rope = attention_with_rope(X_perm, W_Q, W_K, W_V, freqs)

print("=== Permutation equivariance test ===")
print(f"Without RoPE: permuting tokens permutes attention? "
      f"{np.allclose(A_no_rope[perm][:, perm], softmax(X_perm @ W_Q @ (X_perm @ W_K).T / np.sqrt(d_k)))}")
print(f"With RoPE: permuting tokens changes attention patterns? "
      f"{not np.allclose(A_with_rope[perm][:, perm], A_perm_rope)}")
print("\nRoPE breaks permutation equivariance → attention becomes position-aware.")

---
## 9. ALiBi — Attention with Linear Biases

ALiBi adds no positional vectors. Instead, a **linear distance penalty** is added to attention logits:

$$S_{ij} = q_i \cdot k_j - m_h \cdot |i - j|$$

Head slopes $m_h$ form a geometric sequence: $m_h = 2^{-8h/H}$ for $h = 1, \ldots, H$.

In [ ]:
# === ALiBi Implementation ===

def alibi_slopes(n_heads):
    """Compute ALiBi slopes: geometric sequence 2^(-8h/H) for h=1..H."""
    return np.array([2 ** (-8 * h / n_heads) for h in range(1, n_heads + 1)])

def alibi_bias(n, slope):
    """Compute ALiBi bias matrix for one head with given slope."""
    # bias[i,j] = -slope * |i - j|
    positions = np.arange(n)
    return -slope * np.abs(positions[:, None] - positions[None, :])

def attention_with_alibi(X, W_Q, W_K, W_V, slopes, mask=None):
    """Multi-head attention with ALiBi (simplified: one head per slope)."""
    n, d_model = X.shape
    Q = X @ W_Q; K = X @ W_K; V = X @ W_V
    d_k = Q.shape[1]
    
    all_outputs = []
    all_weights = []
    
    for slope in slopes:
        scores = Q @ K.T / np.sqrt(d_k)
        bias = alibi_bias(n, slope)
        scores = scores + bias
        
        if mask is not None:
            scores = scores + mask
        
        weights = softmax(scores)
        all_outputs.append(weights @ V)
        all_weights.append(weights)
    
    return all_outputs, np.array(all_weights)

# Demo
np.random.seed(42)
n_heads = 8
slopes = alibi_slopes(n_heads)

print("=== ALiBi Head Slopes ===")
for h, slope in enumerate(slopes):
    print(f"  Head {h}: slope = {slope:.6f}  (2^(-{8*(h+1)/n_heads:.1f}))")

# Show bias matrices for different slopes
n_demo = 10
print(f"\n=== ALiBi Bias Matrix (n={n_demo}) ===")
if HAS_MPL:
    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    for idx, h in enumerate([0, 2, 5, 7]):
        bias = alibi_bias(n_demo, slopes[h])
        im = axes[idx].imshow(bias, cmap='RdBu_r', vmin=-5, vmax=0)
        axes[idx].set_title(f"Head {h}, slope={slopes[h]:.4f}")
        axes[idx].set_xlabel("Key pos")
        axes[idx].set_ylabel("Query pos")
        plt.colorbar(im, ax=axes[idx], shrink=0.8)
    plt.suptitle("ALiBi bias: steep slopes → local attention, gentle slopes → global attention", fontsize=12)
    plt.tight_layout()
    plt.show()
else:
    for h in [0, 3, 7]:
        bias = alibi_bias(5, slopes[h])
        print(f"Head {h} (slope={slopes[h]:.4f}):")
        print(np.round(bias, 2))

print("\nKey insight: large slopes → rapid distance penalty → head focuses locally")
print("           small slopes → gentle penalty → head can attend far away")

---
## 10. Attention Analysis & Interpretability

### Entropy, attention sinks, induction heads, and rollout

Attention entropy measures how focused or diffuse each head's attention is:
$$H(A_i) = -\sum_j A_{ij} \log A_{ij} \in [0, \log n]$$

In [ ]:
# === Attention Entropy Analysis ===

def attention_entropy(A):
    """
    Compute per-row entropy of attention matrix A.
    H(A_i) = -sum_j A_ij log(A_ij)
    Returns: (n,) array of entropies
    """
    # Clamp to avoid log(0)
    A_safe = np.clip(A, 1e-12, 1.0)
    return -np.sum(A * np.log(A_safe), axis=-1)

def avg_entropy(A):
    """Average entropy across all query positions."""
    return np.mean(attention_entropy(A))

# Example attention distributions
n = 8
print("=== Attention Entropy Examples ===\n")

# Uniform attention
A_uniform = np.ones((n, n)) / n
print(f"Uniform (1/{n}):  H = {avg_entropy(A_uniform):.4f}, max possible = log({n}) = {np.log(n):.4f}")

# Peaked attention (each row attends mostly to one position)
A_peaked = np.eye(n) * 0.97 + (1 - np.eye(n)) * 0.03 / (n - 1)
print(f"Peaked (0.97):   H = {avg_entropy(A_peaked):.4f}")

# One-hot attention
A_onehot = np.eye(n)
print(f"One-hot:          H = {avg_entropy(A_onehot):.6f} (≈0)")

# Causal attention — variable effective sequence length per row
A_causal_demo = np.zeros((n, n))
for i in range(n):
    A_causal_demo[i, :i+1] = softmax(np.random.randn(i+1))
entropies = attention_entropy(A_causal_demo)
print(f"\nCausal attention (random scores):")
for i in range(n):
    max_H = np.log(i + 1) if i > 0 else 0
    norm_H = entropies[i] / max_H if max_H > 0 else 0
    print(f"  Row {i}: H = {entropies[i]:.4f}, max = {max_H:.4f}, normalised = {norm_H:.2%}")

# === Attention Sink Demonstration ===
print("\n=== Attention Sink Phenomenon ===")
print("BOS token (position 0) receives disproportionate attention mass.\n")

# Simulate: make position 0's key vector large
np.random.seed(42)
n, d_k = 12, 16
Q = np.random.randn(n, d_k) * 0.5
K = np.random.randn(n, d_k) * 0.5
K[0] *= 3.0  # BOS key has large norm (common in trained models)

scores = Q @ K.T / np.sqrt(d_k)
A_sink = softmax(scores + causal_mask(n))

# Show attention to position 0 across all query positions
print(f"{'Query pos':>10} | {'Attn to BOS (pos 0)':>20} | {'Max attn to others':>20}")
print("-" * 55)
for i in range(n):
    bos_attn = A_sink[i, 0]
    other_max = A_sink[i, 1:i+1].max() if i > 0 else 0
    marker = " ← sink" if bos_attn > 0.3 else ""
    print(f"{i:>10} | {bos_attn:>20.4f} | {other_max:>20.4f}{marker}")

print("\nBOS consistently receives high attention → 'attention sink'")
print("StreamingLLM exploits this: always keep BOS in cache for stable generation.")

In [ ]:
# === Induction Head Pattern Detection ===
# Pattern: if sequence has [A][B]...[A], induction heads predict [B]
# We simulate this by constructing attention that copies the token after a previous occurrence

np.random.seed(42)

print("=== Induction Head Pattern ===\n")
print("Sequence: [The] [cat] [sat] [on] [the] [mat] [the] [?]")
print("Induction pattern: 'the' appeared at pos 0 → next was 'cat'")
print("                   'the' appeared at pos 4 → next was 'mat'")
print("At pos 6 ('the' again), an induction head would boost 'cat' and 'mat'\n")

# Simulate with token embeddings
vocab = {"the": 0, "cat": 1, "sat": 2, "on": 3, "mat": 4}
tokens = [0, 1, 2, 3, 0, 4, 0]  # the cat sat on the mat the
n_seq = len(tokens)

# Create embeddings where same token = same vector
d = 8
token_embeddings = np.random.randn(5, d) * 0.5  # vocab size = 5
X = np.array([token_embeddings[t] for t in tokens])

# Standard attention
Q = X; K = X  # simplified: Q=K=X (no projections)
scores = Q @ K.T / np.sqrt(d)
A = softmax(scores + causal_mask(n_seq))

print("Attention from pos 6 ('the') to all positions:")
for j in range(n_seq):
    token_name = list(vocab.keys())[tokens[j]]
    marker = " ← same token" if tokens[j] == tokens[6] else ""
    print(f"  → pos {j} ('{token_name}'): {A[6, j]:.4f}{marker}")

print("\nInduction heads specifically learn to:")
print("1. Find matching previous tokens (pos 0 and 4 have same embedding as pos 6)")
print("2. Copy what FOLLOWED those tokens (pos 1='cat' and pos 5='mat')")
print("This requires composition of two heads — a matching head and a copying head.")

In [ ]:
# === Attention Rollout ===
# Approximate information flow through multi-layer attention by multiplying
# adjusted attention matrices: Â_l = 0.5 * A_l + 0.5 * I (residual)

def attention_rollout(attention_matrices):
    """
    Compute attention rollout across layers.
    
    attention_matrices: list of (n, n) attention matrices, one per layer
    Returns: (n, n) rollout matrix showing cumulative information flow
    """
    n = attention_matrices[0].shape[0]
    rollout = np.eye(n)  # start with identity (each token = itself)
    
    for A in attention_matrices:
        # Account for residual connection: 50% from attention, 50% identity
        A_hat = 0.5 * A + 0.5 * np.eye(n)
        # Renormalise rows to sum to 1
        A_hat = A_hat / A_hat.sum(axis=-1, keepdims=True)
        rollout = A_hat @ rollout
    
    return rollout

# Simulate a 4-layer model
np.random.seed(42)
n = 8
n_layers = 4
layer_attentions = []

for l in range(n_layers):
    Q = np.random.randn(n, 16)
    K = np.random.randn(n, 16)
    scores = Q @ K.T / np.sqrt(16) + causal_mask(n)
    layer_attentions.append(softmax(scores))

rollout = attention_rollout(layer_attentions)

print("=== Attention Rollout (4 layers) ===\n")
print("Each entry R[i,j] shows how much input token j")
print("influenced output token i through all 4 layers.\n")

if HAS_MPL:
    fig, axes = plt.subplots(1, 5, figsize=(22, 4))
    for l in range(n_layers):
        im = axes[l].imshow(layer_attentions[l], cmap='Blues', vmin=0, vmax=0.5)
        axes[l].set_title(f"Layer {l+1} attention")
        axes[l].set_xlabel("Key pos")
    
    im = axes[4].imshow(rollout, cmap='Oranges', vmin=0, vmax=0.5)
    axes[4].set_title("Attention Rollout\n(cumulative)")
    axes[4].set_xlabel("Input token")
    axes[4].set_ylabel("Output token")
    plt.colorbar(im, ax=axes[4], shrink=0.8)
    
    plt.suptitle("Individual layer attention vs cumulative rollout", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("Rollout matrix (rows=output, cols=input):")
    print(np.round(rollout, 3))

print(f"\nRollout row sums: {rollout.sum(axis=1).round(4)}")
print("Each output token's influence sums to ~1 (probability distribution over inputs)")

---
## 11. FlashAttention — Tiling Concept

FlashAttention computes **exact** attention without materialising the full $n \times n$ matrix in GPU memory. The key idea: process the attention matrix in **tiles** that fit in fast SRAM, accumulating results with the online softmax trick.

Standard: write $A \in \mathbb{R}^{n \times n}$ to HBM → O(n²) memory  
FlashAttention: process B×B tiles in SRAM → O(n) memory

We implement a simplified CPU version to demonstrate the tiling principle.

In [ ]:
# === FlashAttention-style Tiled Attention ===

def standard_attention(Q, K, V):
    """Standard attention — materialises full n×n matrix."""
    d_k = Q.shape[1]
    S = Q @ K.T / np.sqrt(d_k)  # (n, n) — full matrix in memory
    A = softmax(S)               # (n, n) — full matrix in memory
    return A @ V

def tiled_attention(Q, K, V, block_size=4):
    """
    FlashAttention-style tiled attention.
    
    Processes Q in blocks of size B. For each Q block, iterates over
    K/V blocks and accumulates using the online softmax trick.
    
    Never materialises the full n×n attention matrix — only B×B tiles.
    """
    n, d_k = Q.shape
    d_v = V.shape[1]
    
    O = np.zeros((n, d_v))      # output accumulator
    L = np.zeros(n)             # log-sum-exp accumulator (denominator tracking)
    M = np.full(n, -np.inf)    # running max (for numerical stability)
    
    n_blocks = (n + block_size - 1) // block_size
    
    for i in range(n_blocks):
        # Q block
        q_start = i * block_size
        q_end = min(q_start + block_size, n)
        Q_block = Q[q_start:q_end]  # (B_q, d_k)
        
        for j in range(n_blocks):
            # K, V block
            k_start = j * block_size
            k_end = min(k_start + block_size, n)
            K_block = K[k_start:k_end]  # (B_k, d_k)
            V_block = V[k_start:k_end]  # (B_k, d_v)
            
            # Compute tile scores — only B_q × B_k, not n × n!
            S_tile = Q_block @ K_block.T / np.sqrt(d_k)  # (B_q, B_k)
            
            # Online softmax update (Milakov & Gimelshein 2018)
            for qi in range(q_end - q_start):
                global_qi = q_start + qi
                
                for kj in range(k_end - k_start):
                    score = S_tile[qi, kj]
                    v_vec = V_block[kj]
                    
                    new_max = max(M[global_qi], score)
                    
                    # Rescale previous accumulator
                    old_scale = np.exp(M[global_qi] - new_max)
                    new_scale = np.exp(score - new_max)
                    
                    # Update running sum of exp
                    L[global_qi] = L[global_qi] * old_scale + new_scale
                    
                    # Update output: rescale old + add new contribution
                    O[global_qi] = O[global_qi] * old_scale + new_scale * v_vec
                    
                    M[global_qi] = new_max
    
    # Final normalisation
    for i in range(n):
        O[i] = O[i] / L[i]
    
    return O

# === Verify correctness ===
np.random.seed(42)
n, d_k, d_v = 16, 8, 8
Q = np.random.randn(n, d_k)
K = np.random.randn(n, d_k)
V = np.random.randn(n, d_v)

O_standard = standard_attention(Q, K, V)
O_tiled_4  = tiled_attention(Q, K, V, block_size=4)
O_tiled_8  = tiled_attention(Q, K, V, block_size=8)

print("=== FlashAttention Tiling Verification ===\n")
print(f"n={n}, d_k={d_k}, d_v={d_v}")
print(f"\nStandard vs Tiled (B=4): max error = {np.max(np.abs(O_standard - O_tiled_4)):.2e}")
print(f"Standard vs Tiled (B=8): max error = {np.max(np.abs(O_standard - O_tiled_8)):.2e}")
print(f"Match (atol=1e-10): B=4={np.allclose(O_standard, O_tiled_4, atol=1e-10)}, "
      f"B=8={np.allclose(O_standard, O_tiled_8, atol=1e-10)}")

# Memory comparison
print(f"\n=== Memory Comparison ===")
print(f"Standard attention: stores full {n}×{n} = {n*n} float matrix")
print(f"Tiled (B=4):        stores {4}×{4} = {16} float tile at a time")
print(f"Tiled (B=8):        stores {8}×{8} = {64} float tile at a time")
print(f"Memory reduction:   {n*n}/{16} = {n*n//16}× (B=4)")

# At scale
for n_scale in [1024, 8192, 131072]:
    full_mem = n_scale * n_scale * 2  # fp16 bytes
    tile_mem = 256 * 256 * 2  # typical tile size
    print(f"n={n_scale:>7}: full A = {full_mem/1e9:.2f} GB, tile = {tile_mem/1e6:.2f} MB "
          f"→ {full_mem//tile_mem}× reduction")

---
## 12. Parameter Counting — Real Model Architectures

Exact parameter formulas for the full Transformer block:

$$\text{MHA params} = 4d^2 \quad \text{(standard)}, \quad (2 + 2G/H)d^2 \quad \text{(GQA)}$$
$$\text{FFN params} = 8d^2 \quad \text{(SwiGLU)}, \quad 8d^2 \quad \text{(standard)}$$
$$\text{Total} \approx 12Ld^2 + 2Nd$$

In [ ]:
# === Parameter Counting for Real Models ===

def count_transformer_params(d_model, n_layers, n_heads, n_kv_heads, d_ff, vocab_size,
                              ffn_type="swiglu", has_bias=False, tie_embeddings=False):
    """
    Count exact parameters for a Transformer LM.
    
    Components:
    - Token embeddings: vocab_size × d_model
    - Per layer:
      - MHA: Q, K, V, O projections
      - FFN: up, down (+ gate for SwiGLU)
      - LayerNorm / RMSNorm: 2 × d_model per norm (or d_model for RMSNorm)
    - Final norm: d_model
    - LM head (output projection): d_model × vocab_size (unless tied)
    """
    d_head = d_model // n_heads
    
    # === Embedding ===
    embed_params = vocab_size * d_model
    
    # === Per-layer attention ===
    q_params = d_model * (n_heads * d_head)  # all Q heads
    k_params = d_model * (n_kv_heads * d_head)  # KV heads (GQA)
    v_params = d_model * (n_kv_heads * d_head)
    o_params = (n_heads * d_head) * d_model
    
    if has_bias:
        q_params += n_heads * d_head
        k_params += n_kv_heads * d_head
        v_params += n_kv_heads * d_head
        o_params += d_model
    
    attn_params = q_params + k_params + v_params + o_params
    
    # === Per-layer FFN ===
    if ffn_type == "swiglu":
        # SwiGLU: W_gate (d→d_ff) + W_up (d→d_ff) + W_down (d_ff→d)  = 3 × d × d_ff
        ffn_params = 3 * d_model * d_ff
    else:
        # Standard: W_up (d→d_ff) + W_down (d_ff→d) = 2 × d × d_ff
        ffn_params = 2 * d_model * d_ff
    
    if has_bias:
        ffn_params += d_ff + d_model  # (or more for SwiGLU)
    
    # === Norms: RMSNorm has d_model params (γ only, no β) ===
    norm_params = 2 * d_model  # 2 norms per layer (attn + FFN)
    
    # === Per layer total ===
    per_layer = attn_params + ffn_params + norm_params
    
    # === Model totals ===
    final_norm = d_model
    lm_head = 0 if tie_embeddings else vocab_size * d_model
    
    total = embed_params + n_layers * per_layer + final_norm + lm_head
    
    return {
        "embedding": embed_params,
        "attn_per_layer": attn_params,
        "ffn_per_layer": ffn_params,
        "norm_per_layer": norm_params,
        "per_layer_total": per_layer,
        "all_layers": n_layers * per_layer,
        "final_norm": final_norm,
        "lm_head": lm_head,
        "total": total,
    }

# === Real model configs ===
models = {
    "GPT-2 Small":    dict(d_model=768,   n_layers=12,  n_heads=12, n_kv_heads=12, d_ff=3072,   vocab_size=50257,  ffn_type="standard", has_bias=True),
    "LLaMA-3 8B":     dict(d_model=4096,  n_layers=32,  n_heads=32, n_kv_heads=8,  d_ff=14336,  vocab_size=128256, ffn_type="swiglu"),
    "LLaMA-3 70B":    dict(d_model=8192,  n_layers=80,  n_heads=64, n_kv_heads=8,  d_ff=28672,  vocab_size=128256, ffn_type="swiglu"),
    "Mistral 7B":     dict(d_model=4096,  n_layers=32,  n_heads=32, n_kv_heads=8,  d_ff=14336,  vocab_size=32000,  ffn_type="swiglu"),
}

print("=== Exact Parameter Counts for Real Models ===\n")
print(f"{'Model':<16} | {'Embedding':>12} | {'Attn/Layer':>12} | {'FFN/Layer':>12} | "
      f"{'All Layers':>12} | {'LM Head':>12} | {'Total':>14} | {'Official':>10}")
print("-" * 115)

official = {"GPT-2 Small": 124_000_000, "LLaMA-3 8B": 8_030_000_000, 
            "LLaMA-3 70B": 70_550_000_000, "Mistral 7B": 7_240_000_000}

for name, config in models.items():
    counts = count_transformer_params(**config)
    off = official.get(name, 0)
    pct = f"({counts['total']/off*100:.1f}%)" if off else ""
    
    def fmt(n):
        if n >= 1e9: return f"{n/1e9:.2f}B"
        if n >= 1e6: return f"{n/1e6:.1f}M"
        return f"{n/1e3:.0f}K"
    
    print(f"{name:<16} | {fmt(counts['embedding']):>12} | {fmt(counts['attn_per_layer']):>12} | "
          f"{fmt(counts['ffn_per_layer']):>12} | {fmt(counts['all_layers']):>12} | "
          f"{fmt(counts['lm_head']):>12} | {fmt(counts['total']):>14} | {fmt(off):>10} {pct}")

In [ ]:
# === Attention vs FFN parameter breakdown ===
print("\n=== Attention vs FFN Parameter Breakdown ===\n")

for name, config in models.items():
    counts = count_transformer_params(**config)
    attn_total = counts['attn_per_layer'] * config['n_layers']
    ffn_total = counts['ffn_per_layer'] * config['n_layers']
    total = counts['total']
    
    attn_pct = attn_total / total * 100
    ffn_pct = ffn_total / total * 100
    other_pct = 100 - attn_pct - ffn_pct
    
    print(f"{name}:")
    print(f"  Attention: {attn_total/1e9:.2f}B ({attn_pct:.1f}%)")
    print(f"  FFN:       {ffn_total/1e9:.2f}B ({ffn_pct:.1f}%)")
    print(f"  Other:     {(total - attn_total - ffn_total)/1e9:.2f}B ({other_pct:.1f}%)")
    print()

if HAS_MPL:
    fig, axes = plt.subplots(1, len(models), figsize=(16, 4))
    for idx, (name, config) in enumerate(models.items()):
        counts = count_transformer_params(**config)
        attn_total = counts['attn_per_layer'] * config['n_layers']
        ffn_total = counts['ffn_per_layer'] * config['n_layers']
        other = counts['total'] - attn_total - ffn_total
        
        sizes = [attn_total, ffn_total, other]
        labels = ['Attention', 'FFN', 'Embed+Norm+Head']
        colors = ['#4C72B0', '#DD8452', '#55A868']
        axes[idx].pie(sizes, labels=labels, autopct='%1.1f%%', colors=colors, startangle=90)
        axes[idx].set_title(name, fontsize=11)
    
    plt.suptitle("Parameter distribution: Attention vs FFN vs Other", fontsize=13)
    plt.tight_layout()
    plt.show()
    print("FFN typically has ~2× more parameters than attention (in SwiGLU models).")

---
## 13. Complexity Analysis — Benchmarking the O(n²d) vs O(nd²) Crossover

Total attention complexity: **O(n²d + nd²)**
- n²d term: QKᵀ matrix multiplication + softmax + AV
- nd² term: Q, K, V, O projections

Crossover at n ≈ d: below this, projections dominate; above, the attention matrix dominates.

In [ ]:
# === Complexity Analysis: FLOPs computation ===
import time

def attention_flops(n, d, method="standard"):
    """
    Compute theoretical FLOPs for attention.
    
    Standard: 2n²d (QKᵀ) + 2n² (softmax) + 2n²d (AV) + 8nd² (projections) = 4n²d + 8nd²
    """
    if method == "standard":
        qkt = 2 * n * n * d        # QKᵀ matmul
        softmax_f = 5 * n * n      # exp + sum + div per element (approximate)
        av = 2 * n * n * d          # AV matmul
        proj = 8 * n * d * d        # 4 projection matmuls (Q,K,V,O)
        return {"qkt": qkt, "softmax": softmax_f, "av": av, "proj": proj, 
                "total": qkt + softmax_f + av + proj}
    elif method == "linear":
        # φ(K)ᵀV: nd² + φ(Q)(φ(K)ᵀV): nd²
        return {"total": 4 * n * d * d}  # approximate
    elif method == "sliding_window":
        w = min(d, n)  # window size
        return {"total": 4 * n * w * d + 8 * n * d * d}

# Compare FLOPs across sequence lengths
d = 4096  # typical d_model
print(f"=== Attention FLOPs (d_model = {d}) ===\n")
print(f"{'Seq len':>10} | {'QKᵀ FLOPs':>14} | {'Proj FLOPs':>14} | {'Total':>14} | {'Dominant':>12} | {'% attention':>12}")
print("-" * 90)

for n in [128, 512, 1024, 4096, 8192, 32768, 131072]:
    f = attention_flops(n, d)
    
    attn_compute = f['qkt'] + f['av']
    proj_compute = f['proj']
    dominant = "Attention" if attn_compute > proj_compute else "Projection"
    attn_pct = attn_compute / f['total'] * 100
    
    def fmt_flops(x):
        if x >= 1e15: return f"{x/1e15:.1f} PF"
        if x >= 1e12: return f"{x/1e12:.1f} TF"
        if x >= 1e9:  return f"{x/1e9:.1f} GF"
        if x >= 1e6:  return f"{x/1e6:.1f} MF"
        return f"{x:.0f}"
    
    print(f"{n:>10,} | {fmt_flops(attn_compute):>14} | {fmt_flops(proj_compute):>14} | "
          f"{fmt_flops(f['total']):>14} | {dominant:>12} | {attn_pct:>11.1f}%")

print(f"\nCrossover at n ≈ d = {d}")
print("Below crossover: projections dominate (O(nd²))")
print("Above crossover: attention matrix dominates (O(n²d))")

# Efficiency comparison
print(f"\n=== Method Comparison at n=32768, d={d} ===")
n_long = 32768
methods = {
    "Standard MHA":     attention_flops(n_long, d, "standard")["total"],
    "Linear Attention":  attention_flops(n_long, d, "linear")["total"],
    "Sliding Window":    attention_flops(n_long, d, "sliding_window")["total"],
}

baseline = methods["Standard MHA"]
for name, flops in methods.items():
    ratio = flops / baseline
    print(f"  {name:<20}: {fmt_flops(flops):>12}  ({ratio:.2%} of standard)")

In [ ]:
# === Timing benchmark: O(n²) growth in practice ===
np.random.seed(42)

d_bench = 64  # small d for benchmarking
seq_lengths = [32, 64, 128, 256, 512, 1024]

times_standard = []
times_tiled = []

print("=== Wall-clock timing: standard vs tiled attention ===\n")
print(f"{'n':>6} | {'Standard (ms)':>14} | {'Tiled B=16 (ms)':>16} | {'Ratio':>8} | {'n² growth':>10}")
print("-" * 65)

for n in seq_lengths:
    Q = np.random.randn(n, d_bench)
    K = np.random.randn(n, d_bench)
    V = np.random.randn(n, d_bench)
    
    # Standard
    n_runs = max(1, 5000 // n)
    start = time.time()
    for _ in range(n_runs):
        standard_attention(Q, K, V)
    t_std = (time.time() - start) / n_runs * 1000
    times_standard.append(t_std)
    
    # Tiled
    start = time.time()
    for _ in range(max(1, n_runs // 2)):
        tiled_attention(Q, K, V, block_size=16)
    t_tiled = (time.time() - start) / max(1, n_runs // 2) * 1000
    times_tiled.append(t_tiled)
    
    ratio = t_tiled / t_std if t_std > 0 else 0
    n2_growth = t_std / times_standard[0] if times_standard[0] > 0 else 0
    n2_expected = (n / seq_lengths[0]) ** 2
    
    print(f"{n:>6} | {t_std:>14.3f} | {t_tiled:>16.3f} | {ratio:>7.1f}× | "
          f"{n2_growth:>5.1f}× (expect {n2_expected:.0f}×)")

print("\nNote: tiled version is slower on CPU (Python loops). On GPU with SRAM tiling,")
print("FlashAttention is 2-4× FASTER because it avoids the memory bandwidth bottleneck.")

---
## Summary

| Topic | Key Result | Section |
|-------|-----------|---------|
| Scaled dot-product attention | softmax(QKᵀ/√dₖ)V — numerically stable weighted retrieval | §1 |
| √dₖ scaling | Without scaling, Var(q·k)=dₖ → softmax saturates | §2 |
| Softmax | Stable via max subtraction; Jacobian = diag(p) − ppᵀ | §3 |
| Masking | Causal, padding, sliding window — compose by addition | §4 |
| Multi-Head Attention | H parallel heads, 4d² total params regardless of H | §5 |
| MQA / GQA | Share K,V across heads → H/G × KV cache reduction | §6 |
| KV Cache | 2·n·d·L·bytes — dominates memory at long contexts | §7 |
| RoPE | Rotation gives relative position; dot(R_m q, R_n k) = f(m−n) | §8 |
| ALiBi | Linear distance penalty in attention logits; no PE vectors | §9 |
| Attention analysis | Entropy, sinks, induction heads, rollout | §10 |
| FlashAttention | Exact attention with O(n) memory via SRAM tiling | §11 |
| Parameter counting | ~12Ld² + 2Nd for full Transformer LM | §12 |
| Complexity | O(n²d + nd²); crossover at n ≈ d | §13 |

**Next**: [exercises.ipynb](exercises.ipynb) for hands-on practice, then [FFN Math](../04-FFN-Math/notes.md).